# SymView in Colab

SymView statically analyses a TLA+ specification together with its TLC `.cfg` and proposes
(a) a `SYMMETRY` set -- constants whose values are sets of model values used only through
permutation-equivariant operators -- and (b) a `VIEW` abstraction that drops auxiliary or
history variables.  It never edits the original specification: it emits a wrapper module
`<Module>_SymView.tla` that `EXTENDS` the original, plus one `.cfg` per experimental arm.
TLC then runs each arm and the state counts are recorded.

**Assumption.** The SymView project is already the current working directory of this
notebook (cloned or uploaded beforehand).  This notebook does not clone anything; if you
opened it in Colab, run `%cd <path-to-symview>` first and then execute the cells in order.

**Requirements.** Python 3.10+ and Java 8+ (TLC 2.16).  The setup cell checks both and
prints the Debian/Colab install command if Java is missing.  A full run of the cells below
takes a few minutes; fetching the corpus needs network access.

## 1. Environment

`setup.sh` checks Python >= 3.10, installs `requirements.txt`, verifies the pinned
`tools/tla2tools-1.7.1.jar` against its SHA-256, creates `results/raw` and the analysis
output directories, and runs a static smoke test on `benchmarks/lab/LabLock`.
It is idempotent, so re-running this cell is safe.

In [ ]:
!bash setup.sh

## 2. Fetch the pinned public corpus

The corpus is defined in `config/corpus.yaml` at commit
`ca168055339d5cbcee95b7eefcb97c1ee59f9914` of `tlaplus/Examples` (plus one file from
`ongardie/raft.tla`).  The fetcher records a SHA-256 for every file in
`benchmarks/PROVENANCE.json`; it needs network access.  If `benchmarks/public/` is already
populated, existing files are skipped, but the fetcher still asks GitHub for the pinned
file list, so this cell is not fully offline.

In [ ]:
!python benchmarks/fetch_public_specs.py

## 3. Ground-truth tests

Eight hand-written lab models each isolate one decision (symmetric set, distinguished
element, ordered identifiers, non-equivariant `CHOOSE`, history variable, liveness, ...).
The script checks the tool's verdict against the authored ground truth and should print
`models=8 failures=0`.  It asks TLC for constant values, so it needs Java.

In [ ]:
!python tests/test_lab_ground_truth.py

## 4. Analyse LabLog and run every arm

`LabLog` is a three-process lock with an append-only `log`.  `Proc = {p1, p2, p3}` should be
certified as a symmetry set.  The `log` looks like a history variable, but the checked
`TypeOK` invariant reads it, so a `VIEW` is deliberately refused for this configuration.

The five arms are `direct` (untouched module, original config), `plain` (generated wrapper,
no directive), `symmetry`, `view`, and `both`.  The run writes its full result to JSON so
that the table below is read from evidence rather than retyped.

In [ ]:
%%bash
set -euo pipefail
PYTHONPATH=src python -m symview report --spec-dir benchmarks/lab/LabLog --module LabLog --config LabLog.cfg --arms direct plain symmetry view both --json results/raw/notebook_lablog.json

In [ ]:
import json

import pandas as pd


def arm_table(path):
    with open(path, encoding='utf-8') as handle:
        payload = json.load(handle)
    print('symmetry proposal:', ', '.join(payload['analysis']['symmetry']['selected']) or '-')
    print('view dropped     :', ', '.join(payload['analysis']['history']['dropped']) or '-')
    rows = []
    for arm, run in payload['runs'].items():
        rows.append({
            'arm': arm,
            'verdict': run['verdict'],
            'states generated': run['states_generated'],
            'distinct states': run['distinct_states'],
            'time (s)': run['wall_seconds'],
        })
    frame = pd.DataFrame(rows)
    for column in ('states generated', 'distinct states'):
        frame[column] = frame[column].astype('Int64')
    return frame


arm_table('results/raw/notebook_lablog.json')

### Why `view` is not applicable here

`TypeOK` contains `log \in Seq(LogEntry)`, so dropping `log` would merge states whose
only difference is a log lineage that the invariant can still observe.  Removing `TypeOK`
(`LabLog_notype.cfg`) makes `log` a genuine history variable, and `VIEW` should drop it.
This second run uses the same module with the type invariant removed, so the two tables
show exactly what changes when the property stops reading the history variable.

In [ ]:
%%bash
set -euo pipefail
PYTHONPATH=src python -m symview report --spec-dir benchmarks/lab/LabLog --module LabLog --config LabLog_notype.cfg --arms direct plain symmetry view both --json results/raw/notebook_lablog_notype.json

In [ ]:
arm_table('results/raw/notebook_lablog_notype.json')

## 5. Mapping the notebook to the research questions

- **RQ1 (state/time reduction):** the per-arm state counts above are the raw material;
  the full study aggregates them in `analysis/output/tables/rq1_reduction.md`.
- **RQ2 (violation preservation):** seeded mutants are replayed under each arm and summarised
  in `analysis/output/tables/rq2_violation_preservation.md`.
- **RQ3 (soundness / unsound suggestions):** the 8/8 ground-truth run in section 3 and the
  lab verdict table in `analysis/output/tables/rq3_soundness.md`.
- **RQ4 (ablation symmetry vs view):** the four-arm comparison in
  `analysis/output/tables/rq4_ablation.md`; this notebook already shows `plain`, `symmetry`,
  `view` and `both` for one model.
- **RQ5 (coverage over the corpus):** parser coverage and certified-reduction coverage in
  `analysis/output/tables/parser_robustness.md` and `analysis/output/tables/rq3_soundness.md`.

For the whole pipeline use `bash run_all.sh --quick` (small probe subset, one repeat) or
`bash run_all.sh --full` (the reported study).  All tables and figures are generated from
`results/raw/**`; numbers must not be typed into the report by hand.

Troubleshooting: every command must run from the SymView project root; if Python cannot
import `symview`, prefix the command with `PYTHONPATH=src`.  If TLC fails with
`NoClassDefFoundError`, check that no `CommunityModules.jar` sits next to
`tools/tla2tools-1.7.1.jar` (the jar manifest loads it automatically).